# Forelesning 4: Kalibrering og sammenlikning av resultater

I dag bruker vi statistikken fra forrige forelesning til to vanlige oppgaver i kjemi:

1. **Kalibrering:** Vi lager en kalibreringslinje og bestemmer konsentrasjonen i en ukjent prøve.
2. **Sammenlikning:** Avviker resultatet vårt fra en referanseverdi? Er to grupper eller to analysemetoder forskjellige?

Målet er ikke å samle flest mulig tester, men å velge en metode som svarer på det kjemiske spørsmålet.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"font.size": 14, "figure.figsize": (8, 4.5)})

## Repetisjon (ca. 5 minutter)

<div class="oppgave">

**Prøv selv:** Fila `flame_emission.csv` inneholder emisjonsintensitet ved 589, 671 og 766 nm for prøver med Na⁺, Li⁺ og K⁺.

1. Bruk `groupby` til å beregne antall målinger, gjennomsnitt og standardavvik for emisjonen ved **589 nm** for hvert ion.
2. Legg til en kolonne med RSD i prosent.
3. Hvilket ion gir sterkest signal ved 589 nm? Hvilket har best relativ presisjon?

</div>

In [ ]:
flame = pd.read_csv("../datahandtering_visualisering/data/flame_emission.csv")

summary = flame.groupby(??)[??].agg([??])
summary["rsd_percent"] = ??
summary

<div class="diskuter">

**Diskuter:** Natriumflammen er gul, og den gule natrium-D-linja ligger ved 589 nm. Stemmer det med resultatet? Hvorfor er RSD høyest for ionene som gir svakt signal ved 589 nm?

</div>

## Del 1: Kalibrering

<div class="viktig">

I UV-Vis-spektroskopi sier **Beer-Lamberts lov** at absorbansen er proporsjonal med konsentrasjonen:

$$A = \varepsilon l c$$

Virkelige målinger ligger ikke perfekt på en linje, så vi tilpasser en rett linje til kalibreringsstandardene:

$$A = a c + b$$

En kalibreringsmodell gjelder bare i konsentrasjonsområdet vi har undersøkt.

</div>

### Se på dataene først
`isin` velger radene der kolonnen har én av verdiene i lista.

### Lineær regresjon
`linregress` finner linja som ligger nærmest punktene samlet sett.

<div class="diskuter">

**Diskuter:** Hva betyr stigningstallet og konstantleddet kjemisk? Hvilken enhet har de? Hvorfor bør vi ikke automatisk tvinge linja gjennom origo?

</div>

<div class="viktig">

- **Stigningstallet** er metodens **følsomhet**: hvor mye absorbansen øker per µmol/L. Følsomhet er ikke det samme som presisjon.
- **Konstantleddet** er signalet modellen forventer ved null konsentrasjon, for eksempel fra blankbidrag.
- **R²** beskriver hvor godt punktene følger en rett linje. En høy R² er **ikke** et kvalitetsstempel alene.

</div>

<div class="oppgave">

**Prøv selv:** Tegn modellen sammen med målingene. Fyll inn uttrykket for linja.

</div>

In [ ]:
x_model = np.linspace(0, 10, 100)
y_model = ??

plt.scatter(standards["concentration_uM"], standards["absorbance"], label="Measurements")
plt.plot(x_model, y_model, color="#b23a48", label="Linear model")
plt.xlabel("Concentration (µmol/L)")
plt.ylabel("Absorbance")
plt.legend()
plt.show()

### Bestem den ukjente konsentrasjonen
Vi løser modellen med hensyn på konsentrasjonen:

$$c = \frac{A - b}{a}$$

<div class="oppgave">

**Finn feilen:** Programmet nedenfor gir 0,47 µmol/L. Se på kalibreringsfiguren. Er det rimelig? Finn og rett feilen.

</div>

In [ ]:
unknown = data[data["sample_type"] == "unknown"]
A_unknown = unknown["absorbance"].mean()

c_unknown = A_unknown - result.intercept / result.slope
print(f"Concentration: {c_unknown:.2f} µmol/L")

<div class="viktig">

Dette er et **punktestimat**: vårt beste anslag, ikke et eksakt svar. Både replikatene til den ukjente prøven og selve kalibreringslinja har usikkerhet. Kontroller alltid at signalet ligger **innenfor** kalibreringsområdet.

</div>

<div class="oppgave">

**Prøv selv:** Tipp først: Hva skjer med stigningstallet og konstantleddet hvis vi legger til 0,01 i *alle* absorbansene, for eksempel fordi kyvetten er skitten? Kjør koden og sjekk.

</div>

In [ ]:
shifted = stats.linregress(standards["concentration_uM"], standards["absorbance"] + 0.01)
print(f"Slope: {shifted.slope:.5f}   Intercept: {shifted.intercept:.5f}")

<div class="diskuter">

**Diskuter:** En ukjent prøve gir absorbans 1,2. Hvorfor bør vi være forsiktige med å bruke modellen? Hva bør vi gjøre i laben?

</div>

## Del 2: Stemmer resultatet med en referanseverdi?

Vi analyserer et sertifisert referansemateriale med konsentrasjon 5,00 mg/L. Gjennomsnittet vårt er 5,116 mg/L. Er avviket bare tilfeldig variasjon, eller gir metoden systematisk for høye verdier?

Vi starter med konfidensintervallet. `stats.sem` gir standardfeilen, og `stats.t.interval` gir intervallet direkte.

<div class="diskuter">

**Diskuter:** Referanseverdien 5,00 mg/L ligger utenfor intervallet. Hva forteller det oss?

</div>

### Hypotesetesting

<div class="viktig">

- **Nullhypotesen** $H_0$: Det finnes ingen reell forskjell. Her: $\mu = 5{,}00$ mg/L.
- **Alternativ hypotese** $H_1$: Det finnes en reell forskjell.
- **p-verdien** er sannsynligheten for å få et resultat minst like ekstremt som vårt, **dersom $H_0$ er sann**.

Er $p < 0{,}05$, forkaster vi $H_0$. Statistisk signifikans sier likevel **ikke** hvor stor eller kjemisk viktig forskjellen er.

</div>

<div class="diskuter">

**Diskuter:** Vi fikk $p = 0{,}0007$. Hvilke påstander stemmer?

1. Sannsynligheten for at $H_0$ er sann, er 0,07 %.
2. Hvis metoden egentlig ga riktig verdi, ville så store avvik være svært sjeldne.
3. Avviket er så stort at det har kjemisk betydning.
4. Vi vet nå hvorfor metoden gir for høye verdier.

</div>

<div class="oppgave">

**Prøv selv:** Målingene `[99.8, 100.3, 100.1, 99.9, 100.4]` mg/L skal sammenliknes med en sertifisert verdi på 100,0 mg/L. Beregn konfidensintervallet og utfør en ettutvalgs t-test. Skriv en konklusjon med vanlig språk.

</div>

In [ ]:
measurements = np.array([99.8, 100.3, 100.1, 99.9, 100.4])

ci = stats.t.interval(0.95, ??, loc=??, scale=??)
test = stats.ttest_1samp(??, popmean=??)

print(ci)
print(f"p = {test.pvalue:.3f}")

<div class="diskuter">

**Diskuter:** Betyr $p = 0{,}43$ at vi har *bevist* at metoden gir riktig verdi?

</div>

## Del 3: To uavhengige grupper

Nitrat er målt i seks uavhengige vannprøver fra hver av to innsjøer. Det finnes ingen naturlig kobling mellom en bestemt prøve fra innsjø A og en bestemt prøve fra innsjø B.

Vi bruker `ttest_ind` med `equal_var=False` (**Welchs t-test**), som ikke krever at gruppene har lik spredning.

<div class="oppgave">

**Prøv selv:** Utbyttet fra en syntese er målt i uavhengige forsøk med to katalysatorer. Formuler $H_0$ og $H_1$, og utfør testen. Er forskjellen statistisk signifikant? Er den kjemisk interessant?

- Katalysator A: `[78.2, 79.1, 77.8, 80.0, 78.6]` %
- Katalysator B: `[81.3, 80.7, 82.1, 81.5, 80.9]` %

</div>

In [ ]:
catalyst_A = np.array([78.2, 79.1, 77.8, 80.0, 78.6])
catalyst_B = np.array([81.3, 80.7, 82.1, 81.5, 80.9])

test = ??
print(f"Difference: {??:.2f} %   p = {test.pvalue:.4f}")

## Del 4: To analysemetoder på de samme prøvene

Åtte vannprøver er analysert med både metode A og metode B. Nå hører målingene sammen **parvis**: hver rad er én prøve.

<div class="oppgave">

**Finn feilen:** En student sammenlikner metodene med programmet nedenfor og konkluderer med at det ikke er noen forskjell. Kjør koden. Hva er galt, og hvilken test burde studenten brukt? Sammenlikn p-verdiene.

</div>

In [ ]:
test = stats.ttest_ind(methods["method_B_mg_L"], methods["method_A_mg_L"], equal_var=False)
print(f"p = {test.pvalue:.3f}")

<div class="viktig">

**Paret t-test** (`ttest_rel`) undersøker om gjennomsnittet av **forskjellene innen hvert par** er null. Konsentrasjonen varierer mye fra prøve til prøve, men forskjellen mellom metodene er liten og svært stabil. En uavhengig test blander disse to kildene til variasjon og overser den systematiske forskjellen.

Testen velges ut fra **forsøksdesignet**, ikke ut fra hvilken p-verdi vi ønsker.

</div>

<div class="diskuter">

**Diskuter:** Metode B gir i gjennomsnitt 0,11 mg/L høyere verdier, og $p$ er svært liten. Er det et problem? Hva avhenger svaret av?

</div>

## Hva bruker vi når?

<div class="viktig">

| Kjemisk spørsmål | Metode |
|---|---|
| Hvordan henger signal og konsentrasjon sammen? | Lineær regresjon |
| Avviker én måleserie fra en kjent verdi? | Ettutvalgs t-test (og konfidensintervall) |
| Er to uavhengige grupper forskjellige? | Uavhengig t-test (Welch) |
| Gir to metoder ulike svar på *de samme* prøvene? | Paret t-test |
| Er én mistenkelig verdi uvanlig? | Faglig kontroll, eventuelt Grubbs' test |

</div>

<div class="diskuter">

**Diskuter:** Hvilken metode passer?

1. En ny metode brukes på ti prøver som også er analysert med referansemetoden.
2. Seks målinger av et sertifisert materiale sammenliknes med sertifikatverdien.
3. Fluorescens måles for fem kjente konsentrasjoner.
4. Utbytte med to ulike løsemidler, fem uavhengige synteser med hvert.

</div>

## Del 5: En mistenkelig måling

<div class="viktig">

Før vi regner på en mistenkelig verdi, sjekker vi rådata, enheter, instrumentlogg og labnotater, og vi dokumenterer det vi finner. Vi fjerner **aldri** en verdi bare fordi resultatet ser penere ut uten den.

</div>

**Grubbs' test** undersøker om den mest ekstreme verdien i en omtrent normalfordelt måleserie ligger uvanlig langt fra resten:

$$G = \frac{\text{største avstand fra gjennomsnittet}}{s}$$

**Fyll inn** linja som beregner $G$ i funksjonen nedenfor. Den kritiske verdien er ferdig skrevet, og den formelen trenger du ikke kunne.

In [ ]:
def grubbs(measurements, alpha=0.05):
    """Returnerer beregnet og kritisk G for Grubbs' test (tosidig)."""
    n = len(measurements)
    G = ??
    t_crit = stats.t.ppf(1 - alpha / (2 * n), n - 2)
    G_crit = (n - 1) / np.sqrt(n) * np.sqrt(t_crit**2 / (n - 2 + t_crit**2))
    return G, G_crit

<div class="oppgave">

**Prøv selv:** Beregn gjennomsnitt og standardavvik med og uten 10,42. Hvor mye påvirker verdien resultatet?

</div>

In [ ]:
without = ??

print(f"With:    mean = {np.mean(series):.3f}   SD = {np.std(series, ddof=1):.3f}")
print(f"Without: mean = {??:.3f}   SD = {??:.3f}")

<div class="diskuter">

**Diskuter:** Testen flagger 10,42. Kan vi da fjerne den? Hva mer trenger du å vite? Hva er galt med å kjøre testen på nytt etterpå for å se om det finnes flere utliggere?

</div>

## En enkel arbeidsflyt

<div class="viktig">

1. Formuler det kjemiske spørsmålet
2. Finn ut hva hver observasjon er, og om målingene hører sammen parvis
3. Se på rådataene i en figur
4. Velg metode ut fra spørsmålet og forsøksdesignet
5. Vurder om forutsetningene er rimelige
6. Rapporter både tallet og hva det betyr kjemisk

</div>

Python gjør beregningen. Valget og tolkningen er fortsatt vårt ansvar.

__Oppgaver__:
1. Bruk `uvvis_calibration.csv` til å lage et kalibreringsplott med modellinje og bestemme konsentrasjonen til den ukjente prøven. Skriv en kort rapport med modell, enheter, konsentrasjonsestimat med rimelig antall desimaler og to begrensninger ved resultatet.
2. En student skriver: «R² = 0,999, derfor er konsentrasjonen til den ukjente prøven helt riktig.» Forklar minst tre problemer med denne konklusjonen.
3. Endre én av målingene ved 10 µmol/L, for eksempel fra 0,809 til 0,700. Hva skjer med stigningstallet, R² og konsentrasjonen til den ukjente prøven?
4. Bruk `method_comparison.csv`. Plott forskjellen mellom metodene mot prøvenummer. Ser forskjellen konstant ut, eller avhenger den av konsentrasjonen?
5. Skriv en kort beslutningslogg for måleserien `[10.02, 10.05, 9.98, 10.01, 10.42]` mg/L med observert problem, statistisk resultat, mulig kjemisk forklaring og hvilken informasjon du mangler.

Datafilene ligger i `../datahandtering_visualisering/data/`.